# Unificación automática de CSV 2015–2023

Este notebook consolida archivos CSV anuales

## Lógica principal

1. Detecta los archivos CSV por año.
2. Escanea los encabezados de cada CSV sin cargar todo el archivo.
3. Construye un inventario de columnas por año.
4. Crea la estructura final como la unión de todas las columnas encontradas.
5. Lee todos los CSV conservando los datos como texto.
6. Crea automáticamente las columnas faltantes en cada año.
7. Consolida la información.
8. Exporta la base consolidada en formatos adecuados para muchos registros:
   - CSV
   - Parquet

## Importante
Ahora cualquier columna nueva o faltante se maneja automáticamente desde el inventario de encabezados.


## 1. Instalar requerimientos

In [ ]:
%pip install pandas pyarrow openpyxl

## 2. Importar librerías

In [1]:
from pathlib import Path
import pandas as pd
import csv
import re

## 3. Configuración

Ajusta `CARPETA_CSV` con la ruta donde tienes los archivos.

Ejemplos:

```python
CARPETA_CSV = Path("data")
CARPETA_CSV = Path("../micarpeta")
```


In [ ]:
# Carpeta donde están los CSV originales
CARPETA_CSV = Path("../data_raw")

# Carpeta de salida
CARPETA_SALIDA = Path("../data_raw")
CARPETA_SALIDA.mkdir(exist_ok=True)

# Años a buscar
ANIOS_ESPERADOS = list(range(2015, 2024))

# Separador del CSV.
# Cambia a ";" si tus archivos están separados por punto y coma.
SEPARADOR = ","

# Encoding por defecto, usado solo como respaldo.
ENCODING_DEFAULT = "latin1"

# Archivos de salida para base grande
ARCHIVO_CSV_SALIDA = CARPETA_SALIDA / "base_unificada_estructura_automatica.csv"
ARCHIVO_PARQUET_SALIDA = CARPETA_SALIDA / "base_unificada_estructura_automatica.parquet"

CARPETA_CSV.resolve(), CARPETA_SALIDA.resolve()

(WindowsPath('D:/Intellectus Data/Nacimientos/Nacimientos'),
 WindowsPath('D:/Intellectus Data/Nacimientos/Nacimientos_Consolidados'))

## 4. Funciones generales

In [3]:
def limpiar_nombre_columnas(columnas):
    '''
    Limpia espacios al inicio y final de los nombres de columnas.
    No cambia mayúsculas/minúsculas para respetar la estructura original.
    '''
    return [str(col).strip() for col in columnas]


def detectar_archivos_por_anio(carpeta: Path, anios=ANIOS_ESPERADOS):
    '''
    Busca archivos CSV que contengan el año en el nombre.
    Ejemplo:
    - 2015.csv
    - nacimientos_2015.csv
    - EEVV_2015_final.csv
    '''
    archivos = {}

    for anio in anios:
        patron = re.compile(str(anio))
        encontrados = [
            archivo for archivo in carpeta.glob("*.csv")
            if patron.search(archivo.name)
        ]

        if len(encontrados) == 1:
            archivos[anio] = encontrados[0]

        elif len(encontrados) > 1:
            print(f"Advertencia: hay varios archivos para {anio}:")
            for archivo in encontrados:
                print(" -", archivo)
            archivos[anio] = encontrados[0]
            print(f"Se usará: {encontrados[0]}")

        else:
            print(f"Advertencia: no se encontró archivo para {anio}")

    return archivos

## 5. Detectar archivos CSV por año

In [4]:
archivos_por_anio = detectar_archivos_por_anio(CARPETA_CSV)

print("Archivos detectados:")
for anio, archivo in archivos_por_anio.items():
    print(anio, "->", archivo)

Archivos detectados:
2015 -> Nacimientos\nac2015.csv
2016 -> Nacimientos\nac2016.csv
2017 -> Nacimientos\nac2017.csv
2018 -> Nacimientos\nac2018.csv
2019 -> Nacimientos\nac2019.csv
2020 -> Nacimientos\nac2020.csv
2021 -> Nacimientos\nac2021.csv
2022 -> Nacimientos\nac2022.csv
2023 -> Nacimientos\BD-EEVV-Nacimientos-2023.csv


## 6. Escanear encabezados

Esta parte lee solo la primera fila de cada CSV, por eso es rápida aunque los archivos sean grandes.


In [10]:
def leer_encabezado_csv(ruta_csv: Path):
    '''
    Lee únicamente el encabezado de un CSV.
    Prueba varios encodings para evitar errores UnicodeDecodeError.
    '''
    encodings_a_probar = [ "latin1"]
    ultimo_error = None

    for encoding in encodings_a_probar:
        try:
            encabezado = pd.read_csv(
                ruta_csv,
                sep=SEPARADOR,
                encoding=encoding,
                dtype=str,
                nrows=0
            )

            columnas = limpiar_nombre_columnas(encabezado.columns)
            return columnas, encoding

        except UnicodeDecodeError as e:
            ultimo_error = e

        except Exception as e:
            ultimo_error = e

    raise Exception(f"No fue posible leer el encabezado de {ruta_csv}. Último error: {ultimo_error}")


def construir_inventario_encabezados(archivos_por_anio: dict):
    '''
    Construye un inventario fila a fila:
    ANIO, ARCHIVO, ORDEN_COLUMNA, NOMBRE_COLUMNA, ENCODING_DETECTADO
    '''
    registros = []

    for anio, archivo in archivos_por_anio.items():
        columnas, encoding_usado = leer_encabezado_csv(archivo)

        for posicion, columna in enumerate(columnas, start=1):
            registros.append({
                "ANIO": anio,
                "ARCHIVO": archivo.name,
                "RUTA": str(archivo),
                "ORDEN_COLUMNA": posicion,
                "NOMBRE_COLUMNA": columna,
                "ENCODING_DETECTADO": encoding_usado,
                "TOTAL_COLUMNAS_ARCHIVO": len(columnas)
            })

    return pd.DataFrame(registros)


df_inventario_encabezados = construir_inventario_encabezados(archivos_por_anio)

print("Inventario de encabezados:")
display(df_inventario_encabezados.head(20))

print("\nTotal columnas por año:")
display(
    df_inventario_encabezados
    .groupby("ANIO")["NOMBRE_COLUMNA"]
    .count()
    .reset_index(name="TOTAL_COLUMNAS")
)

Inventario de encabezados:


,ANIO,ARCHIVO,RUTA,ORDEN_COLUMNA,NOMBRE_COLUMNA,ENCODING_DETECTADO,TOTAL_COLUMNAS_ARCHIVO
0,2015,nac2015.csv,Nacimientos\nac2015.csv,1,COD_DPTO,latin1,38
1,2015,nac2015.csv,Nacimientos\nac2015.csv,2,COD_MUNIC,latin1,38
2,2015,nac2015.csv,Nacimientos\nac2015.csv,3,AREANAC,latin1,38
3,2015,nac2015.csv,Nacimientos\nac2015.csv,4,SIT_PARTO,latin1,38
4,2015,nac2015.csv,Nacimientos\nac2015.csv,5,OTRO_SIT,latin1,38
5,2015,nac2015.csv,Nacimientos\nac2015.csv,6,SEXO,latin1,38
6,2015,nac2015.csv,Nacimientos\nac2015.csv,7,PESO_NAC,latin1,38
7,2015,nac2015.csv,Nacimientos\nac2015.csv,8,TALLA_NAC,latin1,38
8,2015,nac2015.csv,Nacimientos\nac2015.csv,9,ANO,latin1,38
9,2015,nac2015.csv,Nacimientos\nac2015.csv,10,MES,latin1,38



Total columnas por año:


,ANIO,TOTAL_COLUMNAS
0,2015,38
1,2016,38
2,2017,38
3,2018,38
4,2019,38
5,2020,37
6,2021,39
7,2022,39
8,2023,39


## 7. Construir estructura final automática

La estructura final se construye con la unión de todos los encabezados encontrados.

No se fuerza ninguna columna manualmente.  
Si un año futuro agrega una columna nueva, esta se detecta y se agrega automáticamente.


In [11]:
def construir_estructura_desde_inventario(df_inventario: pd.DataFrame):
    '''
    Construye la estructura final de columnas a partir del inventario.

    Regla de orden:
    - Se ordena por año y posición original.
    - La primera vez que aparece una columna define su posición.
    - Si años posteriores agregan columnas nuevas, se agregan al final.
    '''
    columnas_finales = []

    inventario_ordenado = df_inventario.sort_values(
        by=["ANIO", "ORDEN_COLUMNA"],
        ascending=[True, True]
    )

    for columna in inventario_ordenado["NOMBRE_COLUMNA"]:
        if columna not in columnas_finales:
            columnas_finales.append(columna)

    # Columna técnica de trazabilidad
    if "ANIO_FUENTE" not in columnas_finales:
        columnas_finales.append("ANIO_FUENTE")

    return columnas_finales


columnas_estructura_final = construir_estructura_desde_inventario(df_inventario_encabezados)

print(f"Total columnas requeridas para unificar: {len(columnas_estructura_final)}")
print(columnas_estructura_final)

Total columnas requeridas para unificar: 40
['COD_DPTO', 'COD_MUNIC', 'AREANAC', 'SIT_PARTO', 'OTRO_SIT', 'SEXO', 'PESO_NAC', 'TALLA_NAC', 'ANO', 'MES', 'ATEN_PAR', 'T_GES', 'T_GES_AGRU_CIE', 'NUMCONSUL', 'TIPO_PARTO', 'MUL_PARTO', 'APGAR1', 'APGAR2', 'IDHEMOCLAS', 'IDFACTORRH', 'IDPERTET', 'EDAD_MADRE', 'EST_CIVM', 'NIV_EDUM', 'ULTCURMAD', 'CODPRES', 'CODPTORE', 'CODMUNRE', 'AREA_RES', 'N_HIJOSV', 'FECHA_NACM', 'N_EMB', 'SEG_SOCIAL', 'IDCLASADMI', 'EDAD_PADRE', 'NIV_EDUP', 'ULTCURPAD', 'PROFESION', 'TIPOFORMULARIO', 'ANIO_FUENTE']


## 8. Matriz de presencia de columnas por año

In [12]:
def construir_matriz_presencia_columnas(df_inventario: pd.DataFrame, columnas_finales: list):
    '''
    Crea una matriz donde cada fila es una columna y cada año indica
    si esa columna existe o no existe en el CSV original.
    '''
    anios = sorted(df_inventario["ANIO"].unique())
    registros = []

    for columna in columnas_finales:
        if columna == "ANIO_FUENTE":
            continue

        registro = {"NOMBRE_COLUMNA": columna}

        for anio in anios:
            columnas_anio = set(
                df_inventario.loc[
                    df_inventario["ANIO"] == anio,
                    "NOMBRE_COLUMNA"
                ]
            )
            registro[str(anio)] = "SI" if columna in columnas_anio else "NO"

        registros.append(registro)

    return pd.DataFrame(registros)


df_matriz_columnas = construir_matriz_presencia_columnas(
    df_inventario_encabezados,
    columnas_estructura_final
)

display(df_matriz_columnas)

,NOMBRE_COLUMNA,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,COD_DPTO,SI,SI,SI,SI,SI,SI,SI,SI,SI
1,COD_MUNIC,SI,SI,SI,SI,SI,SI,SI,SI,SI
2,AREANAC,SI,SI,SI,SI,SI,SI,SI,SI,SI
3,SIT_PARTO,SI,SI,SI,SI,SI,SI,SI,SI,SI
4,OTRO_SIT,SI,SI,SI,SI,SI,SI,SI,SI,SI
5,SEXO,SI,SI,SI,SI,SI,SI,SI,SI,SI
6,PESO_NAC,SI,SI,SI,SI,SI,SI,SI,SI,SI
7,TALLA_NAC,SI,SI,SI,SI,SI,SI,SI,SI,SI
8,ANO,SI,SI,SI,SI,SI,SI,SI,SI,SI
9,MES,SI,SI,SI,SI,SI,SI,SI,SI,SI


## 9. Reporte de columnas faltantes por año

In [13]:
def construir_reporte_faltantes_por_anio(df_inventario: pd.DataFrame, columnas_finales: list):
    '''
    Identifica qué columnas deben crearse vacías en cada año.
    '''
    registros = []
    anios = sorted(df_inventario["ANIO"].unique())

    columnas_finales_sin_trazabilidad = [
        col for col in columnas_finales
        if col != "ANIO_FUENTE"
    ]

    for anio in anios:
        columnas_anio = set(
            df_inventario.loc[
                df_inventario["ANIO"] == anio,
                "NOMBRE_COLUMNA"
            ]
        )

        faltantes = [
            col for col in columnas_finales_sin_trazabilidad
            if col not in columnas_anio
        ]

        registros.append({
            "ANIO": anio,
            "TOTAL_COLUMNAS_ORIGINALES": len(columnas_anio),
            "TOTAL_COLUMNAS_FALTANTES_A_CREAR": len(faltantes),
            "COLUMNAS_FALTANTES_A_CREAR": faltantes
        })

    return pd.DataFrame(registros)


df_reporte_faltantes = construir_reporte_faltantes_por_anio(
    df_inventario_encabezados,
    columnas_estructura_final
)

display(df_reporte_faltantes)

,ANIO,TOTAL_COLUMNAS_ORIGINALES,TOTAL_COLUMNAS_FALTANTES_A_CREAR,COLUMNAS_FALTANTES_A_CREAR
0,2015,38,1,[TIPOFORMULARIO]
1,2016,38,1,[TIPOFORMULARIO]
2,2017,38,1,[TIPOFORMULARIO]
3,2018,38,1,[TIPOFORMULARIO]
4,2019,38,1,[TIPOFORMULARIO]
5,2020,37,2,"[T_GES_AGRU_CIE, TIPOFORMULARIO]"
6,2021,39,0,[]
7,2022,39,0,[]
8,2023,39,0,[]


## 10. Leer todos los CSV como texto

Se usa el encoding detectado durante el escaneo de encabezados.  
Todos los campos se leen como texto para conservar códigos como `01`, `014`, `00`, etc.


In [14]:
def obtener_encoding_por_anio(df_inventario: pd.DataFrame):
    '''
    Obtiene el encoding detectado para cada año.
    '''
    return (
        df_inventario
        .drop_duplicates(subset=["ANIO"])
        .set_index("ANIO")["ENCODING_DETECTADO"]
        .to_dict()
    )


def leer_csv_como_texto(ruta_csv: Path, anio: int, encodings_por_anio: dict):
    '''
    Lee el CSV completo conservando todos los campos como texto.
    '''
    encoding = encodings_por_anio.get(anio, ENCODING_DEFAULT)

    df = pd.read_csv(
        ruta_csv,
        sep=SEPARADOR,
        encoding=encoding,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        low_memory=False
    )

    df.columns = limpiar_nombre_columnas(df.columns)
    df["ANIO_FUENTE"] = str(anio)

    return df


encodings_por_anio = obtener_encoding_por_anio(df_inventario_encabezados)

dataframes = {}

for anio, archivo in archivos_por_anio.items():
    df = leer_csv_como_texto(
        ruta_csv=archivo,
        anio=anio,
        encodings_por_anio=encodings_por_anio
    )

    dataframes[anio] = df
    print(f"{anio}: {df.shape[0]:,} filas x {df.shape[1]:,} columnas leído con {encodings_por_anio.get(anio)}")

2015: 660,999 filas x 39 columnas leído con latin1
2016: 647,521 filas x 39 columnas leído con latin1
2017: 656,704 filas x 39 columnas leído con latin1
2018: 649,115 filas x 39 columnas leído con latin1
2019: 642,660 filas x 39 columnas leído con latin1
2020: 629,402 filas x 38 columnas leído con latin1
2021: 616,914 filas x 40 columnas leído con latin1
2022: 573,625 filas x 40 columnas leído con latin1
2023: 515,549 filas x 40 columnas leído con latin1


## 11. Homologar columnas y consolidar

Para cada año:

- Se comparan sus columnas contra la estructura final automática.
- Se crean vacías las columnas que no existan.
- Se respeta el orden final.
- Se conserva todo como texto.


In [15]:
def homologar_con_estructura_automatica(df: pd.DataFrame, columnas_finales: list, anio: int):
    '''
    Homologa un DataFrame anual contra la estructura final automática.
    '''
    df = df.copy()
    df.columns = limpiar_nombre_columnas(df.columns)

    if "ANIO_FUENTE" not in df.columns:
        df["ANIO_FUENTE"] = str(anio)

    columnas_actuales = list(df.columns)

    columnas_faltantes = [
        col for col in columnas_finales
        if col not in columnas_actuales
    ]

    for col in columnas_faltantes:
        df[col] = ""

    columnas_extra = [
        col for col in columnas_actuales
        if col not in columnas_finales
    ]

    columnas_ordenadas = columnas_finales + columnas_extra
    df = df[columnas_ordenadas]

    reporte = {
        "ANIO": anio,
        "FILAS": len(df),
        "COLUMNAS_ORIGINALES": len(columnas_actuales),
        "COLUMNAS_FINALES": len(df.columns),
        "COLUMNAS_FALTANTES_CREADAS": columnas_faltantes,
        "COLUMNAS_EXTRA_CONSERVADAS": columnas_extra
    }

    return df, reporte


dataframes_normalizados = []
reportes_homologacion = []

for anio, df in dataframes.items():
    df_homologado, reporte = homologar_con_estructura_automatica(
        df=df,
        columnas_finales=columnas_estructura_final,
        anio=anio
    )

    dataframes_normalizados.append(df_homologado)
    reportes_homologacion.append(reporte)

base_unificada = pd.concat(dataframes_normalizados, ignore_index=True)
df_reporte_homologacion = pd.DataFrame(reportes_homologacion)

print("Base consolidada:")
print(f"{base_unificada.shape[0]:,} filas x {base_unificada.shape[1]:,} columnas")

display(df_reporte_homologacion)
display(base_unificada.head())

Base consolidada:
5,592,489 filas x 40 columnas


,ANIO,FILAS,COLUMNAS_ORIGINALES,COLUMNAS_FINALES,COLUMNAS_FALTANTES_CREADAS,COLUMNAS_EXTRA_CONSERVADAS
0,2015,660999,39,40,[TIPOFORMULARIO],[]
1,2016,647521,39,40,[TIPOFORMULARIO],[]
2,2017,656704,39,40,[TIPOFORMULARIO],[]
3,2018,649115,39,40,[TIPOFORMULARIO],[]
4,2019,642660,39,40,[TIPOFORMULARIO],[]
5,2020,629402,38,40,"[T_GES_AGRU_CIE, TIPOFORMULARIO]",[]
6,2021,616914,40,40,[],[]
7,2022,573625,40,40,[],[]
8,2023,515549,40,40,[],[]


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION,TIPOFORMULARIO,ANIO_FUENTE
0,18,001,1,1,,1,6,5,2015,12,...,27/05/2005,4,2,2,44,02,5,1,,2015
1,41,298,1,1,,1,7,5,2015,01,...,25/07/1990,4,1,1,33,02,5,1,,2015
2,41,298,1,1,,2,5,4,2015,10,...,08/02/2008,2,2,2,42,04,11,1,,2015
3,41,298,1,1,,1,4,4,2015,03,...,05/01/2007,3,2,2,39,02,3,1,,2015
4,41,298,1,1,,2,7,5,2015,11,...,12/05/2012,4,1,1,32,04,11,1,,2015


## 12. Validaciones básicas

In [16]:
print("Filas por año:")
display(base_unificada["ANIO_FUENTE"].value_counts().sort_index())

print("\nTotal columnas finales:")
print(len(base_unificada.columns))

print("\nColumnas finales:")
print(base_unificada.columns.tolist())

print("\nTipos de dato:")
display(base_unificada.dtypes.value_counts())

Filas por año:


ANIO_FUENTE
2015    660999
2016    647521
2017    656704
2018    649115
2019    642660
2020    629402
2021    616914
2022    573625
2023    515549
Name: count, dtype: int64


Total columnas finales:
40

Columnas finales:
['COD_DPTO', 'COD_MUNIC', 'AREANAC', 'SIT_PARTO', 'OTRO_SIT', 'SEXO', 'PESO_NAC', 'TALLA_NAC', 'ANO', 'MES', 'ATEN_PAR', 'T_GES', 'T_GES_AGRU_CIE', 'NUMCONSUL', 'TIPO_PARTO', 'MUL_PARTO', 'APGAR1', 'APGAR2', 'IDHEMOCLAS', 'IDFACTORRH', 'IDPERTET', 'EDAD_MADRE', 'EST_CIVM', 'NIV_EDUM', 'ULTCURMAD', 'CODPRES', 'CODPTORE', 'CODMUNRE', 'AREA_RES', 'N_HIJOSV', 'FECHA_NACM', 'N_EMB', 'SEG_SOCIAL', 'IDCLASADMI', 'EDAD_PADRE', 'NIV_EDUP', 'ULTCURPAD', 'PROFESION', 'TIPOFORMULARIO', 'ANIO_FUENTE']

Tipos de dato:


object    40
Name: count, dtype: int64

## 13. Exportar base consolidada

Para bases grandes se evita exportar la base completa a Excel porque Excel solo soporta **1.048.576 filas por hoja**.

Formatos recomendados:

- **CSV:** útil para intercambio.
- **Parquet:** recomendado para análisis, más liviano y eficiente.


In [20]:
# Exportar base consolidada a CSV
base_unificada.to_csv(
    ARCHIVO_CSV_SALIDA,
    index=False,
    encoding=ENCODING_DEFAULT,
    sep=SEPARADOR,
    quoting=csv.QUOTE_MINIMAL
)

# Exportar base consolidada a Parquet
base_unificada.to_parquet(
    ARCHIVO_PARQUET_SALIDA,
    index=False
)

print("Base CSV generada:", ARCHIVO_CSV_SALIDA.resolve())
print("Base Parquet generada:", ARCHIVO_PARQUET_SALIDA.resolve())

Base CSV generada: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\base_unificada_estructura_automatica.csv
Base Parquet generada: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\base_unificada_estructura_automatica.parquet


## 14. Exportar inventarios y reportes

Estos archivos sí se exportan a Excel porque son pequeños y sirven para auditoría.


In [ ]:
ARCHIVO_INVENTARIO_ENCABEZADOS = CARPETA_SALIDA / "inventario_encabezados.csv"
ARCHIVO_MATRIZ_COLUMNAS = CARPETA_SALIDA / "matriz_presencia_columnas.xlsx"
ARCHIVO_REPORTE_FALTANTES = CARPETA_SALIDA / "reporte_columnas_faltantes.xlsx"
ARCHIVO_REPORTE_HOMOLOGACION = CARPETA_SALIDA / "reporte_homologacion_automatica.xlsx"

df_inventario_encabezados.to_csv(
    ARCHIVO_INVENTARIO_ENCABEZADOS,
    index=False,
    encoding=ENCODING_DEFAULT,
    sep=SEPARADOR
)

df_matriz_columnas.to_excel(
    ARCHIVO_MATRIZ_COLUMNAS,
    index=False
)

df_reporte_faltantes.to_excel(
    ARCHIVO_REPORTE_FALTANTES,
    index=False
)

df_reporte_homologacion.to_excel(
    ARCHIVO_REPORTE_HOMOLOGACION,
    index=False
)

print("Inventario encabezados:", ARCHIVO_INVENTARIO_ENCABEZADOS.resolve())
print("Matriz columnas:", ARCHIVO_MATRIZ_COLUMNAS.resolve())
print("Reporte faltantes:", ARCHIVO_REPORTE_FALTANTES.resolve())
print("Reporte homologación:", ARCHIVO_REPORTE_HOMOLOGACION.resolve())

Inventario encabezados: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\inventario_encabezados.csv
Matriz columnas: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\matriz_presencia_columnas.xlsx
Reporte faltantes: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\reporte_columnas_faltantes.xlsx
Reporte homologación: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\reporte_homologacion_automatica.xlsx


## 15. Resumen final

In [19]:
print("Proceso terminado.")
print(f"Filas consolidadas: {base_unificada.shape[0]:,}")
print(f"Columnas consolidadas: {base_unificada.shape[1]:,}")
print("Archivo CSV:", ARCHIVO_CSV_SALIDA.resolve())
print("Archivo Parquet:", ARCHIVO_PARQUET_SALIDA.resolve())

Proceso terminado.
Filas consolidadas: 5,592,489
Columnas consolidadas: 40
Archivo CSV: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\base_unificada_estructura_automatica.csv
Archivo Parquet: D:\Intellectus Data\Nacimientos\Nacimientos_Consolidados\base_unificada_estructura_automatica.parquet
